여러 문서에서 찾아서 답변하는 챗봇 만들기
  - QA Chatbot
  - Langchain
  - ChatGPT
  - ChromaDB

In [7]:
# setting langchain
# 설치 후 세션 반드시 다시 시작
# !pip uninstall -y langchain langchain-core langchain-community langchain-openai
!pip install -q langchain langchain-community langchain-core langchain-openai langchain-text-splitters openai tiktoken chromadb

In [8]:
# OpenAI API 키를 환경 변수에 설정합니다.
# 실제 API 키는 보안상 직접 코드에 넣지 않고 환경 변수나 secrets manager를 사용하는 것이 좋습니다.
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")

In [9]:
# 기사 21개
!wget -q https://github.com/kairess/toy-datasets/raw/master/techcrunch-articles.zip

In [10]:
# 압축 파일 해제
!unzip -q techcrunch-articles.zip -d articles

replace articles/05-07-fintech-space-continues-to-be-competitive-and-drama-filled.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace articles/05-03-databricks-acquires-ai-centric-data-governance-platform-okera.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace articles/05-07-spacex-starship-startups-future.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace articles/05-03-checks-the-ai-powered-data-protection-project-incubated-in-area-120-officially-exits-to-google.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: a
error:  invalid response [a]
replace articles/05-03-checks-the-ai-powered-data-protection-project-incubated-in-area-120-officially-exits-to-google.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: a
error:  invalid response [a]
replace articles/05-03-checks-the-ai-powered-data-protection-project-incubated-in-area-120-officially-exits-to-google.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: a
error:  invalid response [a]
replace articles/05-03-checks-the-ai-powered-data-protection-projec

기본 구조

In [14]:
# =========================================================
# 1. 라이브러리 임포트
# =========================================================

import glob  # 파일 탐색 (여러 txt 로딩용)

from langchain_community.document_loaders import TextLoader  # txt 로더
from langchain_text_splitters import RecursiveCharacterTextSplitter  # chunk 분할
from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # LLM + Embedding

from langchain_community.vectorstores import Chroma  # vector DB

from langchain_core.prompts import ChatPromptTemplate  # prompt 생성
from langchain_core.output_parsers import StrOutputParser  # 문자열 변환
from langchain_core.runnables import RunnablePassthrough  # LCEL passthrough

from langchain_core.runnables import RunnableLambda  # 함수 래핑

from langchain_community.chat_message_histories import ChatMessageHistory  # memory 객체
from langchain_core.runnables.history import RunnableWithMessageHistory  # session memory


# =========================================================
# 2. Memory Store (세션 기반 대화 저장소)
# =========================================================
# 역할: session_id 기준으로 대화 상태 저장 (stateful RAG 핵심)

store = {}   # session_id -> ChatMessageHistory 저장


def get_session_history(session_id: str):
    # 역할: 해당 session의 대화 기록 반환 (없으면 생성)

    if session_id not in store:
        store[session_id] = ChatMessageHistory()   # 새로운 memory 생성

    return  store[session_id]  # 기존 memory 반환


# =========================================================
# 3. LLM + Embedding 모델
# =========================================================

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0  # 재현성 확보
)

embedding = OpenAIEmbeddings(model= "text-embedding-3-small")


# =========================================================
# 4. 문서 로딩 (txt dataset)
# =========================================================

import glob

files = glob.glob("articles/**/*.txt", recursive= True)

print(len(files))  # 모든 txt 파일 수집

docs = []  # 전체 문서 저장

for f in files:
    loader =  TextLoader(f, encoding="utf-8")  # txt 로딩
    docs.extend(loader.load())   # Document 누적


# =========================================================
# 5. Chunking
# =========================================================
# 역할: 긴 문서를 검색 가능한 단위로 분리

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap = 100
)



split_docs = splitter.split_documents(docs)


# =========================================================
# 6. Vector DB 생성 (Chroma)
# =========================================================
# 역할: semantic search 기반 retrieval

vectorstore = Chroma.from_documents(
    split_docs,
    embedding,
    persist_directory="./chroma_db"
)



retriever = vectorstore.as_retriever(search_kwargs = {"k": 4})



# =========================================================
# 7. Query Rewriting (Conversational 핵심)
# =========================================================
# 역할: chat history + question → standalone question

query_rewrite_prompt = ChatPromptTemplate.from_template(
"""
You are a query rewriting assistant.

Chat History:
{chat_history}

Question:
{question}

Rewrite into a standalone question.
"""
)

query_rewriter = query_rewrite_prompt | llm | StrOutputParser()



# =========================================================
# 8. Context Builder (핵심 pipeline)
# =========================================================
# 역할: 1) 질문 재작성 → 2) retrieval → 3) context 생성

def build_context(inputs):

    # (1) query rewriting
    standalone_question = query_rewriter.invoke({
        "chat_history" : inputs["chat_history"],
        "question" : inputs["question"]
    })



    # (2) retrieval
    docs = retriever.invoke(standalone_question)


    # (3) context formatting
    context = "\n\n".join(d.page_content for d in docs)

    return {
        "question" : inputs["question"],
        "chat_history" : inputs["chat_history"],
        "context" : context
    }


# =========================================================
# 9. Prompt (최종 생성용)
# =========================================================

prompt = ChatPromptTemplate.from_template(
"""
You are a Conversational RAG assistant.

Chat History:
{chat_history}

Question:
{question}

Context:
{context}

Answer in Korean:
"""
)


# =========================================================
# 10. Conversational RAG Chain
# =========================================================
# 역할:
# Memory + Query Rewrite + Retrieval + Generation 통합

rag_chain = (
    RunnableLambda(build_context)
    | prompt
    | llm
    | StrOutputParser()

)


# =========================================================
# 11. Memory Wrapper (session 연결)
# =========================================================

rag_with_memory = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key = "question",
    history_messages_key = "chat_history"
)




# =========================================================
# 12. 실행 함수 (테스트용 인터페이스)
# =========================================================

def ask(question, session_id="user-1"):
    """
    역할:
    - Conversational RAG 실행
    - session 기반 memory 유지
    - multi-turn QA 테스트
    """

    result = rag_with_memory.invoke(
        {"question" : question},
        config = {"configurable": {"session_id" : session_id}}
    )



    print("\n==============================")
    print("QUESTION:", question)
    print("\nANSWER:\n", result)
    print("==============================\n")


# =========================================================
# 13. TEST
# =========================================================

ask("이 기사들의 핵심 트렌드는 뭐야?", "user-1")
ask("아까 내용을 더 쉽게 설명해줘", "user-1")
ask("예시 포함해서 정리해줘", "user-1")

21

QUESTION: 이 기사들의 핵심 트렌드는 뭐야?

ANSWER:
 이 기사들의 핵심 트렌드는 다양한 분야(라이프스타일, 스포츠, 엔터테인먼트, 교육 등)에서의 콘텐츠 제공과 함께, 출판 파트너들이 수익을 공유하는 모델을 도입하고 있다는 점입니다. 이는 협력적인 비즈니스 모델을 통해 다양한 콘텐츠를 소비자에게 제공하려는 경향을 나타냅니다.


QUESTION: 아까 내용을 더 쉽게 설명해줘

ANSWER:
 이 기사의 핵심은 GPT-4와 같은 독점 모델들이 많은 주목과 수익을 얻고 있지만, 그들이 가진 자금과 인프라의 우위가 점점 줄어들고 있다는 것입니다. OpenAI의 제품 출시 속도가 빠르게 느껴질 수 있지만, 실제로는 iOS나 포토샵 같은 소프트웨어의 버전 출시와 비교했을 때 여전히 몇 달 또는 몇 년 단위로 이루어지고 있다는 점을 강조하고 있습니다. 즉, 경쟁이 치열해지고 있다는 의미입니다.


QUESTION: 예시 포함해서 정리해줘

ANSWER:
 이 기사의 핵심 내용을 쉽게 정리하면 다음과 같습니다:

1. **경쟁 심화**: GPT-4와 같은 독점 모델들이 주목받고 있지만, 그들의 자금과 인프라의 우위가 줄어들고 있다는 점입니다. 예를 들어, OpenAI의 제품 출시 속도가 빠르게 느껴질 수 있지만, 실제로는 소프트웨어의 버전 출시와 비교했을 때 여전히 몇 달 또는 몇 년 단위로 이루어지고 있습니다. 이는 경쟁이 치열해지고 있다는 것을 의미합니다.

2. **협력적 비즈니스 모델**: 다양한 분야에서 콘텐츠를 제공하고, 출판 파트너들이 수익을 공유하는 모델을 도입하고 있습니다. 예를 들어, 라이프스타일, 스포츠, 엔터테인먼트, 교육 등 여러 분야에서 협력하여 소비자에게 다양한 콘텐츠를 제공하려는 경향이 있습니다.

3. **AI 활용 사례**: 
   - **긴 대화 요약**: 직원들이 긴 대화 스레드를 읽지 않고도 주요 내용을 파악할 수 있도록 도와줍니다. 예를 들어, 여러 메시지가 오간 긴 Slack 대화에서 핵심 포인트를 요약해주

실무
- 문서 로딩

In [15]:
# =========================================================
# 1. 기존 코드 (Original Code)
# =========================================================
# 목적: 디렉토리 내 txt 파일을 로딩하는 기본 방식

# loader = DirectoryLoader('./articles', glob="*.txt", loader_cls=TextLoader)
# documents = loader.load()
# len(documents)


# =========================================================
# 2. 개선 코드 (Improved / Production Ready)
# =========================================================

from langchain_community.document_loaders import DirectoryLoader, TextLoader  # 문서 로더


# =========================================================
# 3. 문서 로딩 (RAG 입력 단계)
# =========================================================
# 개선 포인트:
# - glob 확장 (*.txt → 하위 폴더 포함)
# - path 명시
# - loader_cls 명확화 (확장성 확보)

loader = DirectoryLoader(
    path= "./articles",
    glob= "**/*.txt",
    loader_cls = TextLoader
)


documents = loader.load()  # Document 객체 리스트 생성


# =========================================================
# 4. 데이터 검증 (실무 필수)
# =========================================================
# 개선 포인트:
# - 로딩 성공 여부 즉시 확인
# - RAG 실패 원인 1순위 (empty docs) 방지

print("총 문서 수:", len(documents))


# 샘플 확인 (디버깅용)
if len(documents) > 0:
    print("\n샘플 문서 미리보기:")
    print(documents[0].page_content[:300])

총 문서 수: 21

샘플 문서 미리보기:
AI startup Hugging Face and ServiceNow Research, ServiceNow’s R&D division, have released StarCoder, a free alternative to code-generating AI systems along the lines of GitHub’s Copilot.

Code-generating systems like DeepMind’s AlphaCode; Amazon’s CodeWhisperer; and OpenAI’s Codex, which powers Copi


실무
- chunking

In [16]:

# =========================================================
# 1. 기존 코드 (Original Code)
# =========================================================
# 목적: 문서를 단순히 chunk_size 기준으로 분할
# split texts
# 텍스트 분할기 설정
# chunk_size = 1000 : 텍스트 덩어리의 최대크기를 1000자로 설정(토큰 초과 방지)
# chunk_overlap = 200 : 텍스트 덩어리 간에 200자씩 겹치도록 설정(문맥손실을 줄임)

# text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# texts = text_splitter.split_documents(documents)
# len(texts)


# =========================================================
# 2. 개선 코드 (Improved / Production Ready)
# =========================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter  # 최신 분리 패키지


# =========================================================
# 3. Chunking 설정 (RAG 핵심 성능 파라미터)
# =========================================================
# 개선 포인트:
# - chunk_size: 검색 정확도 vs 문맥 유지 균형
# - chunk_overlap: 문맥 단절 방지 (실무 필수)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200,
    separators= ["\n\n", "\n", ".", "!", "?"]
)




# =========================================================
# 4. 문서 분할 실행
# =========================================================
# 역할: Document → chunk 단위 Document 리스트 변환

texts = text_splitter.split_documents(documents)


# =========================================================
# 5. 결과 검증 (실무 필수)
# =========================================================
# 개선 포인트:
# - chunk 개수 확인 = RAG 검색 난이도 지표
# - 너무 적으면 정보 뭉침 / 너무 많으면 검색 노이즈 증가

print("총 chunk 수:", len(texts))


# 샘플 확인 (디버깅)
if len(texts) > 0:
    print("\nchunk 샘플:")
    print(texts[0].page_content[:300])

총 chunk 수: 233

chunk 샘플:
AI startup Hugging Face and ServiceNow Research, ServiceNow’s R&D division, have released StarCoder, a free alternative to code-generating AI systems along the lines of GitHub’s Copilot.

Code-generating systems like DeepMind’s AlphaCode; Amazon’s CodeWhisperer; and OpenAI’s Codex, which powers Copi


In [17]:
# 실무 디버깅 방식
for i, t in enumerate(texts[2:4]):
    print(f"\n--- chunk {i+2} ---")
    print(t.page_content)


--- chunk 2 ---
Congratulations to all the @BigCodeProject contributors that worked tirelessly over the last 6+ months to bring the vision of releasing a responsibly developed 15B parameter Code LLM to fruition. We cannot thank you enough for the collaboration & contributions to the community. https://t.co/282sCRJq3k — ServiceNow Research (@ServiceNowRSRCH) May 4, 2023

Leandro von Werra, a machine learning engineer at Hugging Face and a co-lead on StarCoder, claims that StarCoder matches or outperforms the AI model from OpenAI that was used to power initial versions of Copilot.

--- chunk 3 ---
“One thing we learned from releases such as Stable Diffusion last year is the creativity and capability of the open-source community,” von Werra told TechCrunch in an email interview. “Within weeks of the release the community had built dozens of variants of the model as well as custom applications. Releasing a powerful code generation model allows anybody to fine-tune and adapt it to their ow

실무
- Chunk Quality 자동 평가 코드 (RAG 디버깅용)
- Chunk quality는 RAG 성능의 60~70%를 결정하는 숨은 핵심 변수다

In [19]:
# =========================================================
# 1. Chunk Quality 자동 평가 코드
# =========================================================
# 목적:
# - chunk가 너무 짧거나 긴지 확인
# - 문장 깨짐 여부 간단 점검
# - RAG 입력 품질 사전 검증

import numpy as np
import re


# =========================================================
# 2. 기본 통계 계산 함수
# =========================================================

def evaluate_chunks(chunks):
    lengths = [len(c.page_content) for c in chunks]

    print("===================================")
    print("Chunk Quality Report")
    print("===================================\n")

    # 기본 통계
    print(f"총 chunk 수: {len(chunks)}")
    print(f"평균 길이: {np.mean(lengths):.2f}")
    print(f"최소 길이: {np.min(lengths)}")
    print(f"최대 길이: {np.max(lengths)}")
    print(f"표준편차: {np.std(lengths):.2f}")


    # =========================================================
    # 3. 너무 짧은 chunk (노이즈 가능)
    # =========================================================
    short_chunks = [c for c in chunks if len(c.page_content) < 200]

    # =========================================================
    # 4. 너무 긴 chunk (검색 비효율)
    # =========================================================
    long_chunks = [c for c in chunks if len(c.page_content) > 1200]


    print("\n===================================")
    print("Quality Issues")
    print("===================================\n")

    print(f"너무 짧은 chunk (<200): {len(short_chunks)}")
    print(f"너무 긴 chunk (>1200): {len(long_chunks)}")


    # =========================================================
    # 5. 문장 깨짐 간단 체크 (heuristic)
    # =========================================================
    broken = 0

    for c in chunks:
        text = c.page_content.strip()


        # 문장 끝이 비정상적인 경우 체크
        if not text.endswith((".", "!", "?", "\"")):
            broken += 1

        # 너무 많은 공백 or 깨진 텍스트
        if len(re.findall(r"[가-힣a-zA-Z0-9]", text)) < 20:
            broken += 1

    print(f"\n의심 chunk (문장 깨짐 가능): {broken}")


    # =========================================================
    # 6. 샘플 출력
    # =========================================================
    print("\n===================================")
    print("Sample Chunk Preview")
    print("===================================\n")

    for i, c in enumerate(chunks[:3]):
        print(f"[Chunk {i}]")
        print(c.page_content[:300])
        print("\n---\n")


# =========================================================
# 7. 실행
# =========================================================

evaluate_chunks(texts)

Chunk Quality Report

총 chunk 수: 233
평균 길이: 760.18
최소 길이: 232
최대 길이: 998
표준편차: 171.46

Quality Issues

너무 짧은 chunk (<200): 0
너무 긴 chunk (>1200): 0

의심 chunk (문장 깨짐 가능): 57

Sample Chunk Preview

[Chunk 0]
AI startup Hugging Face and ServiceNow Research, ServiceNow’s R&D division, have released StarCoder, a free alternative to code-generating AI systems along the lines of GitHub’s Copilot.

Code-generating systems like DeepMind’s AlphaCode; Amazon’s CodeWhisperer; and OpenAI’s Codex, which powers Copi

---

[Chunk 1]
According to a study from the University of Cambridge, at least half of developers’ efforts are spent debugging and not actively programming, which costs the software industry an estimated $312 billion per year. But so far, only a handful of code-generating AI systems have been made freely available

---

[Chunk 2]
Congratulations to all the @BigCodeProject contributors that worked tirelessly over the last 6+ months to bring the vision of releasing a responsibly developed 1

Create Chroma DB
  1. text-> Embeddings
  2. db 폴더에 데이터 저장
  3. DB 초기화
  4. db 폴더로부터 DB로드

In [ ]:
# attr버전 상의 오류가 발생되면 업데이트 해준다

!pip install -qU chromadb attrs langchain langchain-core langchain-community langchain-openai

In [20]:
"""
구조흐름

[1] texts
 → embedding
[2] Chroma.from_documents()
 → db 저장
[3] persist()
 → disk 저장
[4] Chroma(persist_directory=...)
 → 다시 로드
 """

# =========================================================
# 1. 기존 코드 (Original Code)
# =========================================================
# 목적: texts → embedding → Chroma DB 생성 및 저장

# persist_directory = 'db'
# embedding = OpenAIEmbeddings()
# vectordb = Chroma.from_documents(
#     documents=texts,
#     embedding=embedding,
#     persist_directory=persist_directory
# )


# =========================================================
# 2. 개선 코드 (Improved / Production Ready)
# RAG 검색 엔진의 index 생성 단계
# =========================================================

from langchain_openai import OpenAIEmbeddings  # embedding model
from langchain_community.vectorstores import Chroma  # vector DB


# =========================================================
# 3. DB 저장 경로 설정
# =========================================================
# 역할: Chroma vector DB가 로컬에 저장되는 위치 지정

persist_directory =  "db"    # DB persistence folder


# =========================================================
# 4. Embedding 모델 초기화
# =========================================================
# 역할: 텍스트 → 벡터 변환 (RAG 핵심 단계)
# OpenAI 임베딩 모델을 초기화
# 이 모델은 텍스트를 벡터(숫자 배열)로 변환하는 역할


embedding = OpenAIEmbeddings(
    model=  "text-embedding-3-small"   # 최신 안정 모델
)


# =========================================================
# 5. Chroma DB 생성 (Indexing 단계)
# =========================================================
# 역할:
# texts → embedding → vector DB 생성 + 디스크 저장
# 분할된 텍스트 덩어리(texts)와 임베딩 모델(embedding)을 사용하여 Chroma 벡터 데이터베이스를 생성
# persist_directory에 데이터베이스 파일이 저장

vectordb = Chroma.from_documents(
    documents=texts,            # chunked documents
    embedding=embedding,        # embedding function
    persist_directory=  persist_directory   # DB 저장 경로
)


# =========================================================
# 6. DB 생성 확인 (실무 필수)
# =========================================================
# 역할: indexing 성공 여부 확인

print("DB 생성 완료")
print("저장 위치:", persist_directory)
print("문서 수:", len(texts))


DB 생성 완료
저장 위치: db
문서 수: 233


In [21]:
# =========================================================
# 1. 기존 코드 (Original Code)
# =========================================================
# 목적: 생성된 벡터 DB를 디스크에 저장하고 메모리 해제

# vectordb.persist()
# vectordb = None

# =========================================================
# 2. DB 디스크 저장 (Persistence)
# =========================================================
# 역할: 메모리 기반 vector DB → 로컬 디스크로 저장

vectordb.persist()   # Chroma DB를 disk에 저장


# =========================================================
# 3. 메모리 해제 (Resource Optimization) -> 안하면 RAM 누적 → Out of memory 발생
# =========================================================
# 역할: RAM 사용량 줄이기 (대규모 RAG 필수)

vectordb = None   # 메모리 객체 제거


# =========================================================
# 4. 저장 확인 (실무 디버깅용)
# =========================================================
print("Chroma DB 저장 완료 (persist 완료)")
print("vectordb 메모리 해제 상태:", vectordb)

Chroma DB 저장 완료 (persist 완료)
vectordb 메모리 해제 상태: None


/tmp/ipykernel_3462/2365224006.py:14: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()   # Chroma DB를 disk에 저장


In [22]:
# =========================================================
# 1. DB 로드 (Persistence → Memory)
# =========================================================
# 역할:
# disk(db 폴더)에 저장된 벡터 DB를 다시 불러와서 사용

vectordb = Chroma(
    persist_directory = persist_directory,
    embedding_function= embedding
)



# =========================================================
# 2. 로드 확인 (실무 디버깅 필수)
# =========================================================
print("Chroma DB 로드 완료")
print("persist_directory:", persist_directory)

Chroma DB 로드 완료
persist_directory: db


/tmp/ipykernel_3462/4171145132.py:7: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(


Retriever 생성

In [23]:
# =========================================================
# 1. Retriever 생성 (Vector DB → Search Engine)
# =========================================================
# 역할:
# Chroma DB를 "검색 가능한 인터페이스"로 변환

retriever = vectordb.as_retriever()


# =========================================================
# 2. (선택) 검색 성능 설정 - 실무 필수
# =========================================================
# k = Top-K 검색 개수 (중요)

retriever = vectordb.as_retriever(
    search_kwargs={"k": 4}  # 상위 4개 chunk 검색
)


In [24]:
# 검색기를 사용하여 'What is Generative AI?' 쿼리와 관련된 문서를 검색
# =========================================================
# 1. Query 설정
# =========================================================
# 역할:
# query → vector search → 관련 document 반환

query = "What is Generative AI?"


# =========================================================
# 2. Retriever 실행 (NEW API)
# =========================================================
# 역할: query → 관련 문서 검색

docs = retriever.invoke(query)



# =========================================================
# 2. 결과 출력 (디버깅용)
# =========================================================
# 역할:
# 어떤 문서에서 답을 가져왔는지 확인 (RAG 핵심 디버깅)

for i, doc in enumerate(docs):
    print("=" * 50)
    print(f"[Result {i}]")
    print("Source : ", doc.metadata.get("source", "No source metadata"))
    print("Content Preview:")
    print(doc.page_content[:300])



[Result 0]
Source :  articles/05-04-slack-updates-aim-to-put-ai-at-the-center-of-the-user-experience.txt
Content Preview:
Developers can get in on the action too, building AI steps into workflows, giving them the option of tapping into external apps and large language models to build generative AI experiences themselves. Just last week the company made its updated developer experience generally available, and this shou
[Result 1]
Source :  articles/05-04-microsoft-doubles-down-on-ai-with-new-bing-features.txt
Content Preview:
Generative art AI has been in the headlines a lot, lately — and not for the most optimistic of reasons necessarily.

Plaintiffs have brought several lawsuits against OpenAI and its rival vendors, alleging that copyrighted data — mostly art — was used without their permission to train generative mode
[Result 2]
Source :  articles/05-03-spawning-lays-out-its-plans-for-letting-creators-opt-out-of-generative-ai-training.txt
Content Preview:
The legal spats between art

In [27]:
# =========================================================
# 1. 기존 방식 (문제 발생 구조 - 참고용)
# =========================================================
# dict → retriever 전달 구조에서 타입 에러 발생
#
# rag_chain = (
#     RunnablePassthrough.assign(
#         context=retrieval_chain
#     )
#     | llm
# )

# 문제 원인:
# - retriever가 dict 입력을 받음
# - LLM에 dict 그대로 전달되는 경우 발생
# - tiktoken error / type error 발생


# =========================================================
# 2. 개선된 안정 구조 (LCEL 표준 방식)
# =========================================================

from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser


# =========================================================
# 3. Retriever 안전 래핑 (핵심 수정)
# =========================================================
# 역할: dict → string 변환 후 retriever 전달

def retrieve(inputs):
    # inputs는 {"question": "..."} 형태
    query = inputs["question"]
    return retriever.invoke(query)


retriever_runnable = RunnableLambda(retrieve)


# =========================================================
# 4. 문서 포맷팅 함수
# =========================================================
# 역할: Document 리스트 → 하나의 문자열로 변환

def format_docs(docs):

    return "\n\n".join(doc.page_content for doc in docs)


format_runnable = RunnableLambda(format_docs)


# =========================================================
# 5. LLM 설정
# =========================================================

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0
)


# =========================================================
# 6. Retrieval Chain (검색 단계)
# =========================================================
# 역할: question → docs

retrieval_chain = retriever_runnable | format_runnable


# =========================================================
# 7. Prompt 생성 함수
# =========================================================
# 역할: context + question → LLM 입력 포맷 생성

def build_prompt(x):
    return f"""
You are a helpful AI assistant.

Use the context below to answer.

Context:
{x['context']}

Question:
{x['question']}
"""


prompt_runnable = RunnableLambda(build_prompt)


# =========================================================
# 8. 최종 RAG Chain (안정 구조)
# =========================================================
# 핵심 구조:
# input(dict)
#   → context 생성 (retriever)
#   → prompt 생성
#   → LLM
#   → output

rag_chain = (
    RunnablePassthrough.assign(
        context = lambda x : retrieval_chain.invoke({
            "question" : x["question"]
        })
    )
    | prompt_runnable
    | llm
    | StrOutputParser()
)


# =========================================================
# 9. 실행 테스트
# =========================================================

query = "What is Generative AI?"

result = rag_chain.invoke({
    "question": query
})

print("\n==============================")
print("QUESTION:", query)
print("\nANSWER:\n", result)
print("==============================")


QUESTION: What is Generative AI?

ANSWER:
 Generative AI refers to artificial intelligence models that "learn" to create art, code, and other content by "training" on sample images and text, typically scraped from the public web. These models are used to generate new content based on the patterns and information they have learned during the training process.


In [28]:
# 결과를 k개 반환
# retiever를 재 설정해서 반환할 문서의 개수를 k개로 설정
# (search_kwargs={'k':3} : 가장 관련성 높은 문서 3개 반환

# =========================================================
# 1. 기존 코드 (기본 Retriever 생성 방식)
# =========================================================
# 역할: Vector DB에서 retriever를 기본 설정으로 생성
#
# retriever = vectordb.as_retriever()


# =========================================================
# 2. 개선 코드 (Top-K 제어 추가)
# =========================================================
# 역할: 검색 결과 개수(k)를 명시적으로 설정하여
#       RAG 품질을 제어하는 방식

retriever = vectordb.as_retriever(
    search_kwargs={
        "k": 3  # 상위 3개 문서만 반환 (Top-K retrieval)
    }
)

In [29]:

# =========================================================
# 1. 기존 코드 (Deprecated 방식)
# =========================================================
# LangChain 최신 버전에서는 사용 불가
#
# docs = retriever.get_relevant_documents("What is Generative AI?")


# =========================================================
# 2. 개선 코드 (최신 LangChain 방식)
# =========================================================
# 역할: retriever에서 관련 문서 Top-K 검색 수행
# - invoke()는 LCEL 표준 인터페이스

query = "What is Generative AI?"

docs = retriever.invoke(query)


# =========================================================
# 3. 검색 결과 출력 (출처 확인)
# =========================================================
# 역할: RAG 디버깅용 (어떤 문서에서 가져왔는지 확인)

for doc in docs:
    print(doc.metadata.get("source", "source 없음"))

articles/05-04-slack-updates-aim-to-put-ai-at-the-center-of-the-user-experience.txt
articles/05-04-microsoft-doubles-down-on-ai-with-new-bing-features.txt
articles/05-03-spawning-lays-out-its-plans-for-letting-creators-opt-out-of-generative-ai-training.txt


Chain 생성

In [30]:

# =========================================================
# 1. 기존 방식 (LCEL 구조 - 현재 환경에서 오류 발생)
# =========================================================
#  retriever | format_docs
#  dict LCEL pipeline
#  function + pipe 혼합
#
# 문제:
# - LangChain 버전/환경에 따라 graph 변환 실패
# - TypeError 반복 발생
# - 디버깅 어려움


# =========================================================
# 2. 개선 방식 (직렬 RAG - 안정 구조)
# =========================================================
# 역할: Retriever → Format → Prompt → LLM 순서로 직접 실행

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser


# =========================================================
# 3. LLM 설정
# =========================================================
# 역할: 최종 답변 생성 모델

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0
)


# =========================================================
# 4. Retriever 실행 함수
# =========================================================
# 역할: query → 관련 문서 검색 (Top-K)

def retrieve_docs(query):
    return retriever.invoke(query)


# =========================================================
# 5. 문서 포맷 함수
# =========================================================
# 역할: Document 리스트 → LLM 입력용 문자열 변환

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# =========================================================
# 6. RAG Pipeline (직렬 구조)
# =========================================================
# 전체 흐름:
# query → retriever → docs → format → prompt → LLM → answer

def rag_pipeline(query):

    # 1) 문서 검색
    docs = retrieve_docs(query)

    # 2) 문서 문자열 변환
    context = format_docs(docs)

    # 3) 프롬프트 생성
    prompt = f"""
You are a helpful assistant.

Use the context below to answer the question.

Context:
{context}

Question:
{query}
"""

    # 4) LLM 실행
    response = llm.invoke(prompt)

    # 5) 결과 반환
    return response.content


# =========================================================
# 7. 실행 테스트
# =========================================================

query = "What is Generative AI?"

result = rag_pipeline(query)

print("\n==============================")
print("QUESTION:", query)
print("\nANSWER:\n", result)
print("==============================")


QUESTION: What is Generative AI?

ANSWER:
 Generative AI refers to artificial intelligence models that "learn" to create art, code, and more by "training" on sample images and text, usually scraped indiscriminately from the web.


In [31]:

# =========================================================
# 1. 기존 방식 (RetrievalQA 기반 출력 구조)
# =========================================================
# llm_response가 dict 형태일 때만 동작
#
# llm_response['result']
# llm_response['source_documents']
#
# 문제:
# - LCEL / invoke 기반 구조에서는 사용 불가
# - 반환 타입이 str 또는 AIMessage로 바뀜


# =========================================================
# 2. 개선 방식 (현재 RAG 구조 기준)
# =========================================================
# 역할: LLM 응답 + source 문서 보기 좋게 출력

def process_llm_response(llm_response, docs=None):
    """
    llm_response: LLM 결과 (str 또는 AIMessage)
    docs: retriever에서 가져온 문서 (source 출력용)
    """

    # =========================================================
    # 1) LLM 결과 출력
    # =========================================================
    # LCEL / invoke 기준: content 존재 여부 처리
    try :
        print(llm_response.content)

    except:
        print(llm_response)

    # =========================================================
    # 2) Source 출력 (RAG 검증용)
    # =========================================================
    print("\n\nSources:")

    if docs:
        for source in docs:
            print(source.metadata.get("source", "no source metadata"))

Query

In [ ]:

# =========================================================
# 1. Query 설정
# =========================================================

query = "How much money did Pando raise?"


# =========================================================
# 2. Retriever 실행
# =========================================================

docs =


# =========================================================
# 3. Context 생성
# =========================================================

context =


# =========================================================
# 4. LLM 호출
# =========================================================

prompt = f"""
Use the context below to answer the question.

Context:
{context}

Question:
{query}
"""

llm_response =


# =========================================================
# 5. 결과 출력 (호환 함수 사용)
# =========================================================

process_llm_response(llm_response, docs)

NameError: name 'retriever' is not defined

In [ ]:
print(llm_response.content)

Pando raised $30 million in a Series B round, bringing its total raised to $45 million.


In [ ]:

query = "What is Generative AI?"

# 1. 검색
docs = retriever.invoke(query)

# 2. context 생성
context = format_docs(docs)

# 3. LLM 실행
llm_response = llm.invoke(f"""
Use the context below.

Context:
{context}

Question:
{query}
""")

# 4. 출력
print(llm_response.content)

# 5. source 확인
for doc in docs:
    print(doc.metadata.get("source", "no source"))

Generative AI refers to artificial intelligence technology that is capable of creating new content, such as art, music, or text, based on patterns and data it has been trained on. This type of AI "learns" to generate new content by analyzing and processing existing data, often scraped from the internet, and then creating new content that mimics the patterns and styles of the original data.
articles/05-04-slack-updates-aim-to-put-ai-at-the-center-of-the-user-experience.txt
articles/05-04-microsoft-doubles-down-on-ai-with-new-bing-features.txt
articles/05-03-spawning-lays-out-its-plans-for-letting-creators-opt-out-of-generative-ai-training.txt


In [ ]:
query = 'CMA가 뭐야? 한글로 답변해줘.'

# 1. 검색
docs = retriever.invoke(query)

# 2. context 생성
context = format_docs(docs)

# 3. LLM 실행
llm_response = llm.invoke(f"""
Use the context below.

Context:
{context}

Question:
{query}
""")

# 4. 출력
print(llm_response.content)

# 5. source 확인
for doc in docs:
    print(doc.metadata.get("source", "no source"))

CMA는 영국의 경쟁 및 시장 규제 기관인 Competition and Markets Authority의 약자입니다.
articles/05-04-cma-generative-ai-review.txt
articles/05-04-cma-generative-ai-review.txt
articles/05-04-cma-generative-ai-review.txt


Text Splitter
  - RAG 시스템의 두번째 단계로 로드된 문서를 효율적으로 처리하고 시스템이 정보를 보다 잘 활용할 수 있도록 준비하는 과정
  - LLM이 받아들일 수 있는 효율적인 작은 규모의 조각으로 나누는 작업
  - 문서분할 과정
    1. 문서 구조 파악
    2. 단위 설정
    3. chunk_size
    4. chunk_overlap

In [ ]:
"""
RAG 전체 흐름에서 위치
1. Load Documents
2. Text Splitter  ← 여기
3. Embedding
4. Vector DB 저장
5. Retrieval
6. LLM 생성
"""

'\nRAG 전체 흐름에서 위치\n1. Load Documents\n2. Text Splitter  ← 여기\n3. Embedding\n4. Vector DB 저장\n5. Retrieval\n6. LLM 생성\n'

In [ ]:
# 설치 후 세션 재시작
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 84.8 MB/s eta 0:00:00


In [ ]:
!pip install -q pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 5.2 MB/s eta 0:00:00


In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

# =========================================================
# 1. 기존 방식 (LangChain PDF + Embedding 준비)
# =========================================================
# 단순 문자열 split 기반 chunking은 토큰 기준 아님
# OpenAI embedding만 사용하면 한국어 성능 저하 가능


# =========================================================
# 2. 라이브러리 임포트
# =========================================================

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# 기존
# from langchain.embeddings import HuggingFaceEmbeddings

# 수정 (정상)
from langchain_community.embeddings import HuggingFaceEmbeddings

import tiktoken
import pypdf


# =========================================================
# 3. PDF 로딩
# =========================================================
# 역할: PDF → Page 단위 Document 변환

loader = PyPDFLoader(
    "/content/drive/MyDrive/AI_ChatBot/LLM_원본/자료/[이슈리포트 2022-2호] 혁신성장 정책금융 동향.pdf"
)

pages =       # 로드 + 분할을 한 번에


# =========================================================
# 4. Tokenizer 설정 (중요)
# =========================================================
# 역할: GPT 모델 기준 토큰 길이 계산

tokenizer =


def tiktoken_len(text):
    # 텍스트 → 토큰 변환 → 길이 반환
    tokens =

    return len(tokens)


# =========================================================
# 5. Text Splitter (토큰 기반 chunking)
# =========================================================
# 기존 방식 vs 개선 방식
#
# character 기반 chunking
# token 기반 chunking (정확도 ↑)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,          # 토큰 기준 chunk 크기
    chunk_overlap=0,         # overlap 없음 (명확한 분리)
    length_function=   # 핵심: token 기반 길이 계산
)

docs = text_splitter.split_documents(pages)


# =========================================================
# 6. 한국어 Embedding 모델
# =========================================================
# 역할: 문장을 vector로 변환 (한국어 최적화)

model_name = "jhgan/ko-sbert-nli"

model_kwargs = {
      # GPU 없을 때 안전 설정
}

encode_kwargs = {
      # cosine similarity 안정화
}

ko = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

/tmp/ipykernel_786/3484624517.py:82: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  ko = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.46k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/538 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:

# =========================================================
# 7. FAISS Vector DB 생성
# =========================================================
# 역할: 문서(chunk)를 embedding해서 검색 가능한 DB로 변환

from langchain_community.vectorstores import FAISS

vectordb = FAISS.from_documents(
    documents=docs,
    embedding=ko
)


# =========================================================
# 8. Retriever 생성
# =========================================================
# 역할: 질문 → 관련 문서 검색

retriever = vectordb.as_retriever(search_kwargs={"k": 3})


# =========================================================
# 9. 검색 테스트
# =========================================================

query = "이 보고서의 핵심 정책은 무엇인가?"

results = retriever.invoke(query)


# =========================================================
# 10. 결과 출력
# =========================================================
# 역할: 검색된 chunk 내용 확인

print("\n================ 검색 결과 ================\n")

for i, doc in enumerate(results):
    print(f"[Chunk {i+1}]")
    print(doc.page_content[:500])  # 너무 길어서 500자만 출력
    print("\nSOURCE:", doc.metadata.get("source", "no source"))
    print("-" * 60)


================ 검색 결과 ================

[Chunk 1]
혁신성장 정책금융 동향 : ICT 산업을 중심으로
  CIS이슈리포트 2022-2호 | 5 |

SOURCE: /content/drive/MyDrive/AI_ChatBot/LLM_원본/자료/[이슈리포트 2022-2호] 혁신성장 정책금융 동향.pdf
------------------------------------------------------------
[Chunk 2]
혁신성장 정책금융 동향 : ICT 산업을 중심으로
  CIS이슈리포트 2022-2호 | 15 |

SOURCE: /content/drive/MyDrive/AI_ChatBot/LLM_원본/자료/[이슈리포트 2022-2호] 혁신성장 정책금융 동향.pdf
------------------------------------------------------------
[Chunk 3]
혁신성장 정책금융 동향 : ICT 산업을 중심으로
  CIS이슈리포트 2022-2호 | 1 |

SOURCE: /content/drive/MyDrive/AI_ChatBot/LLM_원본/자료/[이슈리포트 2022-2호] 혁신성장 정책금융 동향.pdf
------------------------------------------------------------


CharacterTextSplitter
  - 가장 간단한 텍스트 분할기, 특정 구분자를 기준으로 텍스트를 분할

In [ ]:
# 로컬 파일에서 텍스트 읽기
with open('/content/drive/MyDrive/AI_ChatBot/LLM_원본/자료/state_of_the_union.txt') as f:
  state_of_the_union = f.read()

In [ ]:

# =========================================================
# 1. CharacterTextSplitter 개념
# =========================================================
# 역할: 가장 단순한 텍스트 분할 방식
# 기준: 문장 의미가 아니라 "문자 / 구분자 기준"


# =========================================================
# 2. 기존 방식 vs CharacterTextSplitter 특징
# =========================================================
# RecursiveCharacterTextSplitter
#   → 의미 기반 분할 (실전 RAG 추천)
#
# CharacterTextSplitter
#   → 단순 문자열 기준 분할
#   → 빠르지만 문맥 깨질 수 있음


# =========================================================
# 3. CharacterTextSplitter 초기화
# =========================================================

from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(

    # -----------------------------------------
    # separator
    # -----------------------------------------
    # 역할: 텍스트를 나누는 기준
    # 여기서는 "문단 단위 분리"
    separator=  ,

    # -----------------------------------------
    # chunk_size
    # -----------------------------------------
    # 역할: 하나의 chunk 최대 길이
    chunk_size=  ,

    # -----------------------------------------
    # chunk_overlap
    # -----------------------------------------
    # 역할: chunk 간 문맥 유지용 겹치는 영역
    chunk_overlap=  ,

    # -----------------------------------------
    # length_function
    # -----------------------------------------
    # 역할: chunk 크기 계산 기준
    # 기본: len() → 문자 수 기준
    length_function=
)

In [ ]:

# =========================================================
# 1. 텍스트 분할 실행
# =========================================================
# 역할: 입력 텍스트(state_of_the_union)를 chunk로 분할

texts = text_splitter.split_text(state_of_the_union)


# =========================================================
# 2. chunk 개수 확인
# =========================================================
# 역할: 전체 문서가 몇 개 chunk로 나뉘었는지 확인

print("총 chunk 개수:", len(texts))


# =========================================================
# 3. chunk 내용 확인 (샘플)
# =========================================================

print("\n================ Chunk 1 ================\n")
print(texts[0])

print("\n=========================================\n")

print("\n================ Chunk 2 ================\n")
print(texts[1])

print("\n=========================================\n")

print("\n================ Chunk 3 ================\n")
print(texts[2])


# =========================================================
# 4. chunk 길이 확인 (품질 체크 핵심)
# =========================================================
# 역할: chunk size 설정이 제대로 적용됐는지 검증

print("\n================ Chunk Length ================\n")

print("Chunk 1 길이:", len(texts[0]))
print("Chunk 2 길이:", len(texts[1]))
print("Chunk 3 길이:", len(texts[2]))

총 chunk 개수: 55

================ Chunk 1 ================

Madame Speaker, Vice President Biden, members of Congress, distinguished guests, and fellow Americans:

Our Constitution declares that from time to time, the president shall give to Congress information about the state of our union. For 220 years, our leaders have fulfilled this duty. They have done so during periods of prosperity and tranquility. And they have done so in the midst of war and depression; at moments of great strife and great struggle.



================ Chunk 2 ================

It's tempting to look back on these moments and assume that our progress was inevitable, that America was always destined to succeed. But when the Union was turned back at Bull Run and the Allies first landed at Omaha Beach, victory was very much in doubt. When the market crashed on Black Tuesday and civil rights marchers were beaten on Bloody Sunday, the future was anything but certain. These were times that tested the courage of our c

In [ ]:

# =========================================================
# 1. chunk 길이 분석 목적
# =========================================================
# 역할: 각 chunk가 얼마나 균일하게 나뉘었는지 확인
# 중요성: RAG 검색 품질에 직접 영향


# =========================================================
# 2. chunk 길이 리스트 생성
# =========================================================
# 역할: 각 chunk의 문자 길이를 리스트로 저장

char_list = []

for i in range(len(texts)):
    char_list.append(len(texts[i]))


# =========================================================
# 3. 결과 출력
# =========================================================

print(char_list)

[454, 707, 930, 848, 896, 818, 846, 892, 813, 721, 730, 919, 996, 926, 839, 900, 850, 529, 614, 493, 582, 746, 1163, 878, 483, 607, 740, 419, 902, 794, 714, 416, 894, 917, 934, 711, 548, 506, 784, 948, 619, 669, 510, 1015, 714, 527, 701, 597, 436, 750, 558, 992, 920, 786, 790]


In [ ]:
# 문서를 Document 객체로 생성하고 분할
text_splitter.create_documents([state_of_the_union])

[Document(metadata={}, page_content='Madame Speaker, Vice President Biden, members of Congress, distinguished guests, and fellow Americans:\n\nOur Constitution declares that from time to time, the president shall give to Congress information about the state of our union. For 220 years, our leaders have fulfilled this duty. They have done so during periods of prosperity and tranquility. And they have done so in the midst of war and depression; at moments of great strife and great struggle.'),
 Document(metadata={}, page_content="It's tempting to look back on these moments and assume that our progress was inevitable, that America was always destined to succeed. But when the Union was turned back at Bull Run and the Allies first landed at Omaha Beach, victory was very much in doubt. When the market crashed on Black Tuesday and civil rights marchers were beaten on Bloody Sunday, the future was anything but certain. These were times that tested the courage of our convictions and the strengt

RecursiveCharacterTextSplitter
  - 재귀적으로 문서를 분할
  - '\n\n'을 기준으로 문서를 분할한 후 여전히 청크 크기가 큰 경우 '\n'을 기준으로 문서를 분할한다

In [ ]:

# =========================================================
# 1. RecursiveCharacterTextSplitter 개념
# =========================================================
# 역할: 의미 구조를 최대한 유지하면서 텍스트를 분할하는 splitter
# 특징: 문단 → 문장 → 문자 순으로 "재귀적으로" 분할


# =========================================================
# 2. Character vs Recursive 차이
# =========================================================
# CharacterTextSplitter
#   → 단순 문자 기준 분할 (문맥 깨짐 가능)
#
# RecursiveCharacterTextSplitter
#   → 문단 → 문장 → 문자 순으로 단계적 분할
#   → 의미 보존 능력 높음 (RAG 표준)


# =========================================================
# 3. RecursiveCharacterTextSplitter 설정
# =========================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(

    # -----------------------------------------
    # chunk_size
    # -----------------------------------------
    # 역할: 하나의 chunk 최대 길이
    chunk_size=1000,

    # -----------------------------------------
    # chunk_overlap
    # -----------------------------------------
    # 역할: chunk 간 문맥 유지용 겹치는 길이
    chunk_overlap=100,

    # -----------------------------------------
    # length_function
    # -----------------------------------------
    # 역할: chunk 길이 계산 기준
    # 기본: len() → 문자 기준
    length_function=len
)

In [ ]:
# RAG에서 chunking 결과가 제대로 나왔는지 확인하는 필수 디버깅 코드
# =========================================================
# 1. 텍스트 분할 실행
# =========================================================
# 역할: 원본 문서를 chunk 단위로 분할
# 전제: state_of_the_union 변수에 긴 텍스트가 존재해야 함

texts = text_splitter.split_text(state_of_the_union)


# =========================================================
# 2. chunk 개수 확인
# =========================================================
# 역할: 전체 문서가 몇 개 chunk로 나뉘었는지 확인

print("총 chunk 개수:", len(texts))


# =========================================================
# 3. chunk 내용 확인 (샘플 출력)
# =========================================================

print("\n================ Chunk 1 ================\n")
print(texts[0])

print("\n" + "-" * 100 + "\n")

print("\n================ Chunk 2 ================\n")
print(texts[1])

print("\n" + "-" * 100 + "\n")

print("\n================ Chunk 3 ================\n")
print(texts[2])


# =========================================================
# 4. chunk 길이 확인 (품질 분석 핵심)
# =========================================================
# 역할: chunk size 설정이 제대로 적용됐는지 검증

print("\n================ Chunk Length ================\n")

print("Chunk 1 길이:", len(texts[0]))
print("Chunk 2 길이:", len(texts[1]))
print("Chunk 3 길이:", len(texts[2]))

총 chunk 개수: 57

================ Chunk 1 ================

Madame Speaker, Vice President Biden, members of Congress, distinguished guests, and fellow Americans:

Our Constitution declares that from time to time, the president shall give to Congress information about the state of our union. For 220 years, our leaders have fulfilled this duty. They have done so during periods of prosperity and tranquility. And they have done so in the midst of war and depression; at moments of great strife and great struggle.

----------------------------------------------------------------------------------------------------


================ Chunk 2 ================

It's tempting to look back on these moments and assume that our progress was inevitable, that America was always destined to succeed. But when the Union was turned back at Bull Run and the Allies first landed at Omaha Beach, victory was very much in doubt. When the market crashed on Black Tuesday and civil rights marchers were beaten on 

In [ ]:

# =========================================================
# 1. chunk 길이 분석 목적
# =========================================================
# 역할: 각 chunk의 문자 길이를 수치로 수집하여 분포를 분석
# 목적: RAG chunking 품질 평가 (균일성 / 이상치 확인)


# =========================================================
# 2. 결과 저장 리스트 초기화
# =========================================================

char_list = []


# =========================================================
# 3. 각 chunk 길이 계산
# =========================================================
# 역할: texts 리스트를 순회하면서 각 chunk의 길이 측정

for i in range(len(texts)):
    char_list.append(len(texts[i]))


# =========================================================
# 4. 결과 출력
# =========================================================
# 역할: chunk 길이 분포 확인

print(char_list)

[454, 707, 930, 848, 896, 818, 846, 892, 813, 721, 730, 919, 996, 926, 839, 900, 850, 529, 614, 493, 582, 746, 995, 259, 878, 483, 607, 740, 419, 902, 794, 714, 416, 894, 917, 934, 711, 548, 506, 784, 948, 619, 669, 510, 996, 116, 714, 527, 701, 597, 436, 750, 558, 992, 920, 786, 790]


In [ ]:
import numpy as np

print("평균:", np.mean(char_list))
print("최대:", np.max(char_list))
print("최소:", np.min(char_list))
print("표준편차:", np.std(char_list))

평균: 722.438596491228
최대: 996
최소: 116
표준편차: 196.40327448800113


In [ ]:

# =========================================================
# 1. Embedding 기반 Chunk Quality 평가 목적
# =========================================================
# 역할: chunk가 의미적으로 잘 구성되었는지 자동 평가
# 기준:
#   - chunk 내부 의미 일관성 (coherence)
#   - chunk 간 중복 여부 (diversity)
#   - chunk 길이 안정성 (stability)


# =========================================================
# 2. 필요한 라이브러리
# =========================================================

from langchain_openai import OpenAIEmbeddings  # embedding 모델
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity  # cosine similarity


# =========================================================
# 3. Embedding 모델 초기화
# =========================================================

embeddings =


# =========================================================
# 4. chunk embedding 생성
# =========================================================
# 역할: 각 chunk를 vector로 변환

chunk_vectors =


# =========================================================
# 5. Chunk 내부 coherence 평가
# =========================================================
# 역할: 하나의 chunk가 의미적으로 얼마나 일관된지 평가
# 방법: chunk를 반으로 나누고 cosine similarity 계산

def split_half(text):
    # chunk를 단순히 반으로 분할 (proxy 방식)


    return


coherence_scores = []

for chunk in texts:

    # chunk를 두 부분으로 나눔
    a, b =

    # 각 부분 embedding 생성
    vec_a =

    vec_b =

    # cosine similarity 계산 (의미 일관성)
    sim =

    coherence_scores.append(sim)


# 평균 coherence 출력
print("Chunk 내부 coherence 평균:", np.mean(coherence_scores))


# =========================================================
# 6. Chunk 간 similarity 분석 (중복 여부)
# =========================================================
# 역할: chunk들끼리 얼마나 비슷한지 확인

similarity_matrix =

# upper triangle만 사용 (중복 제거)

upper_vals =

print("Chunk 간 평균 similarity:", np.mean(upper_vals))
print("Chunk 간 최대 similarity:", np.max(upper_vals))


# =========================================================
# 7. Chunk 길이 분석 (stability)
# =========================================================
# 역할: chunk size 균일성 확인

lengths = [len(t) for t in texts]

avg_len = np.mean(lengths)
std_len = np.std(lengths)

print("평균 chunk 길이:", avg_len)
print("chunk 길이 표준편차:", std_len)


# =========================================================
# 8. 최종 Chunk Quality Score (heuristic)
# =========================================================
# 역할: 3가지 지표를 합쳐 하나의 quality score 생성

quality_score = (
            # 내부 일관성 (가중치 50%)
            # 중복 낮을수록 좋음 (30%)
                 # 길이 안정성 (20%)
)


print("\n==============================")
print("FINAL CHUNK QUALITY SCORE:", quality_score)
print("==============================")

Chunk 내부 coherence 평균: 0.8219946741521498
Chunk 간 평균 similarity: 0.8028036654122412
Chunk 간 최대 similarity: 0.9216128093475993
평균 chunk 길이: 722.438596491228
chunk 길이 표준편차: 196.40327448800113

FINAL CHUNK QUALITY SCORE: 0.6157839280431596


In [ ]:
!pip install -q langchain-community pypdf

In [ ]:

# =========================================================
# 1. PDF Loader 임포트 (최신 LangChain 기준)
# =========================================================
# 역할: PDF 파일을 페이지 단위 Document 객체로 변환

from langchain_community.document_loaders import PyPDFLoader


# =========================================================
# 2. PDF 파일 경로 설정
# =========================================================
# 역할: Google Drive 또는 로컬 PDF 경로 지정

pdf_path = "/content/drive/MyDrive/AI_ChatBot/LLM_원본/자료/[이슈리포트 2022-2호] 혁신성장 정책금융 동향.pdf"


# =========================================================
# 3. PDF Loader 생성
# =========================================================

loader = PyPDFLoader(pdf_path)


# =========================================================
# 4. PDF 로딩 + 페이지 단위 분할
# =========================================================
# 역할: PDF → Document 리스트 (page 단위로 자동 split)

pages = loader.load_and_split()


# =========================================================
# 5. 결과 확인
# =========================================================

print("총 페이지 수:", len(pages))
print("\n첫 번째 페이지 미리보기:\n")
print(pages[0].page_content[:500])

총 페이지 수: 18

첫 번째 페이지 미리보기:

혁신성장 정책금융 동향 : ICT 산업을 중심으로
  CIS이슈리포트 2022-2호 | 1 |
<요  약>▶혁신성장 정책금융기관*은 혁신성장산업 영위기업을 발굴·지원하기 위한 정책금융 가이드라인**에 따라 혁신성장 기술분야에 대한 금융지원을 강화하고 있음     * 산업은행, 기업은행, 수출입은행, 신용보증기금, 기술보증기금, 중소벤처기업진흥공단, 무역보험공사 등 11개 기관    ** 혁신성장 정책금융 지원 대상을 판단하는 기준으로, ‘9대 테마 – 46개 분야 – 296개 품목’으로 구성￮정책금융기관의 혁신성장 정책금융 공급규모는 2017년 24.1조 원에서 2021년 85.4조 원으로 크게 증가하여 국내 산업 구조의 미래 산업으로의 전환을 충실히 지원하고 있음￮본 보고서는 ICT 산업의 정책금융 지원 트렌드를 파악하고, 혁신성장 정책금융이 집중되는 주요 품목의 기술·시장 동향을 분석함▶혁신성장 ICT 산업은 정보통신(6개 분야, 47개 품목), 전기전자(5개 분야, 27개 품목


In [ ]:
# 1번 페이지 내용 출력
print(pages[1].page_content)

| 2 | CIS이슈리포트 2022-2호 
▶혁신성장 ICT 산업의 정책금융 공급규모 및 공급속도를 종합적으로 분석한 결과, 차세대무선통신미디어, 능동형컴퓨팅(이상 정보통신 테마), 차세대반도체(전기전자 테마) 및 객체탐지(센서측정 테마) 기술분야로 혁신성장 정책금융이 집중되고 있음[ICT 산업 내 주요 기술분야 혁신성장 정책금융 공급 현황]                                                            (단위: 억 원, %)테마(대분류) 주요 기술분야(중분류) 정책금융 공급규모연평균 공급액 증가율(%)테마 내 공급 점유율(%)2017년 말2021년 말정보통신차세대무선통신미디어7,82027,86537.445.1능동형컴퓨팅35216,032159.810.1전기전자차세대반도체12,01953,77945.458.5센서측정객체탐지1,2786,71151.448.5▶주요 기술분야별 세부 품목단위로는 5G 이동통신시스템, 인공지능(AI), 시스템반도체 및 스마트센서에 정책금융 공급량이 높은 것으로 확인됨￮정부가 미래 먹거리산업으로 선정한 인공지능(AI)의 미래성장율(CAGR: 41.0%)이 가장 높으며, 시장규모는 시스템반도체(3,833.8억 달러, 2025년)가 가장 큰 것으로 분석됨￮4대 품목은 공통적으로 수요기반이 크고, 각국 정부가 중점적으로 육성을 지원하고 있어 시장이 지속 성장할 것으로 전망되나, 원천기술 미확보 및 높은 해외 의존도가 약점으로 지적되어 국내 기업의 경쟁력 강화가 시급한 것으로 평가됨[혁신성장 ICT 주요 품목 시장전망]                                                            (단위: 억 달러, %)주요 기술분야(중분류)주요 품목(소분류) 시장규모 전망시장 촉진·저해요인2020년2025년(E)CAGR(%)차세대무선통신미디어5G이동통신시스템494.41,982.032.0Ÿ(촉진) 정부의 국제표준 확보 의지Ÿ(저해) 소재에 대한 높은 해외 의존도능동형컴퓨팅인공지능

In [ ]:

# =========================================================
# 1. RecursiveCharacterTextSplitter 개념
# =========================================================
# 역할: 문서를 의미 단위를 최대한 유지하면서 chunk로 분할
# 특징: 문단 → 문장 → 문자 순으로 재귀적으로 분할 (RAG 표준 방식)


# =========================================================
# 2. Import
# =========================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter


# =========================================================
# 3. Text Splitter 설정
# =========================================================

text_splitter = RecursiveCharacterTextSplitter(

    # -----------------------------------------
    # chunk_size
    # -----------------------------------------
    # 역할: 하나의 chunk 최대 길이 (문자 기준)
    chunk_size=1000,

    # -----------------------------------------
    # chunk_overlap
    # -----------------------------------------
    # 역할: chunk 간 문맥 유지용 겹침 영역
    # 중요: 문장 끊김 방지 + embedding 품질 향상
    chunk_overlap=200,

    # -----------------------------------------
    # length_function
    # -----------------------------------------
    # 역할: chunk 길이 계산 기준 함수
    # 기본: len → 문자 수 기준
    length_function=len
)

In [ ]:
# 문서를 설정된 분할기로 분할
texts = text_splitter.split_documents(pages)

In [ ]:
# 분할된 텍스트 덩어리 개수 출력
len(texts)

37

In [ ]:
# 각 텍스트 덩어리의 문자 길이 리스트 생성 및 출력
char_list = []
for i in range(len(texts)):
  char_list.append(len(texts[i].page_content))
print(char_list)

[783, 22, 962, 552, 52, 997, 280, 22, 999, 376, 688, 642, 52, 946, 699, 534, 52, 999, 439, 23, 995, 433, 53, 999, 221, 522, 665, 503, 435, 633, 53, 998, 411, 936, 975, 913, 95]


토큰단위 텍스트 분할기
  - 텍스트 분할의 목적은 LLM이 소화할 수 있을 정도의 텍스트만 호출하도록 만드는 것이다
  - LLM은 텍스트를 받아들일 때, 정해진 토큰 이상으로 소화할 수 없기때문에 글을 토큰 단위로 분할한다면 최대한 많은 글을 포함하도록 청크를 분할할 수 있다

In [ ]:

# =========================================================
# 1. tiktoken 설치
# =========================================================
# 역할: OpenAI 모델 기준 토큰 단위로 텍스트를 인코딩/디코딩하기 위한 라이브러리
# 용도: chunk size를 "문자"가 아니라 "token" 기준으로 제어할 때 사용

!pip install tiktoken

In [ ]:

# =========================================================
# 1. tiktoken 임포트
# =========================================================
# 역할: OpenAI GPT 모델의 토큰 단위 인코딩/디코딩 라이브러리

import tiktoken


# =========================================================
# 2. tokenizer 초기화
# =========================================================
# 역할: GPT 모델에서 사용하는 표준 인코딩 방식 설정
# cl100k_base = GPT-3.5 / GPT-4 계열 공통 tokenizer

tokenizer =


# =========================================================
# 3. 토큰 길이 계산 함수 정의
# =========================================================
# 역할: 문자열을 token 단위로 변환 후 길이를 반환
# 중요성: RAG chunk size를 "token 기준"으로 설정할 때 사용

def tiktoken_len(text):

    # text → token list 변환
    tokens = tokenizer.encode(text)

    # token 개수 반환
    return len(tokens)

In [ ]:
print(texts[2].page_content)

▶혁신성장 ICT 산업의 정책금융 공급규모 및 공급속도를 종합적으로 분석한 결과, 차세대무선통신미디어, 능동형컴퓨팅(이상 정보통신 테마), 차세대반도체(전기전자 테마) 및 객체탐지(센서측정 테마) 기술분야로 혁신성장 정책금융이 집중되고 있음[ICT 산업 내 주요 기술분야 혁신성장 정책금융 공급 현황]                                                            (단위: 억 원, %)테마(대분류) 주요 기술분야(중분류) 정책금융 공급규모연평균 공급액 증가율(%)테마 내 공급 점유율(%)2017년 말2021년 말정보통신차세대무선통신미디어7,82027,86537.445.1능동형컴퓨팅35216,032159.810.1전기전자차세대반도체12,01953,77945.458.5센서측정객체탐지1,2786,71151.448.5▶주요 기술분야별 세부 품목단위로는 5G 이동통신시스템, 인공지능(AI), 시스템반도체 및 스마트센서에 정책금융 공급량이 높은 것으로 확인됨￮정부가 미래 먹거리산업으로 선정한 인공지능(AI)의 미래성장율(CAGR: 41.0%)이 가장 높으며, 시장규모는 시스템반도체(3,833.8억 달러, 2025년)가 가장 큰 것으로 분석됨￮4대 품목은 공통적으로 수요기반이 크고, 각국 정부가 중점적으로 육성을 지원하고 있어 시장이 지속 성장할 것으로 전망되나, 원천기술 미확보 및 높은 해외 의존도가 약점으로 지적되어 국내 기업의 경쟁력 강화가 시급한 것으로 평가됨[혁신성장 ICT 주요 품목 시장전망]                                                            (단위: 억 달러, %)주요 기술분야(중분류)주요 품목(소분류) 시장규모 전망시장 촉진·저해요인2020년2025년(E)CAGR(%)차세대무선통신미디어5G이동통신시스템494.41,982.032.0Ÿ(촉진) 정부의 국제표준 확보 의지Ÿ(저해) 소재에 대한 높은 해외


In [ ]:
tiktoken_len(texts[2].page_content)

888

문제) 빈칸 채우기

In [ ]:

# =========================================================
# 1. Token 기반 RecursiveCharacterTextSplitter 설정
# =========================================================
# 역할: 문서를 chunk로 나눌 때 "토큰 기준"으로 길이를 계산
# 기존 char 기준이 아니라 GPT 기준(context window 기준)으로 분할


text_splitter = RecursiveCharacterTextSplitter(

    # -----------------------------------------
    # chunk_size
    # -----------------------------------------
    # 역할: 하나의 chunk 최대 길이 (token 기준으로 계산됨)
    chunk_size=   ,

    # -----------------------------------------
    # chunk_overlap
    # -----------------------------------------
    # 역할: chunk 간 겹침 제거 (0이면 완전 독립 chunk)
    chunk_overlap=   ,

    # -----------------------------------------
    # length_function
    # -----------------------------------------
    # 역할: chunk 길이 계산 기준 함수
    # 여기서는 tiktoken 기반 token 수 사용
    length_function=
)


# =========================================================
# 2. PDF Document → Token 기반 chunk 분할
# =========================================================
# 역할: pages (PDF page Document 리스트)를 RAG용 chunk로 변환

texts =

In [ ]:
print(len(texts[2].page_content))
print(tiktoken_len(texts[2].page_content))

956
881


In [ ]:
token_list = []
for i in range(len(texts)):
  token_list.append(tiktoken_len(texts[i].page_content))
print(token_list)

[795, 19, 881, 460, 51, 994, 134, 19, 936, 179, 539, 577, 51, 890, 393, 564, 51, 998, 111, 19, 936, 99, 985, 522, 723, 382, 360, 659, 51, 932, 96, 818, 846, 19, 980, 78]
